# pde-nondim — worked examples

**Symbolic non-dimensionalisation of PDEs with automatic group identification and PINN code generation.**

```
pip install pde-nondim
```

This notebook walks through five examples of increasing complexity:

| # | PDE | Groups found |
|---|-----|--------------|
| 1 | Heat equation | Fourier number |
| 2 | Burgers equation (nonlinear) | Reynolds number |
| 3 | Advection-diffusion | Péclet number |
| 4 | Nonlinear diffusion with `k(u)` | 1/Pe with nonlinear placeholder |
| 5 | LPBF 3-D Goldak heat source | Pe_c, δ_x, δ_h, γ + PINN code |

In [1]:
import sympy as sp
from pde_nondim import NonDimensionalizer, MultipleScalings
from pde_nondim.autoscale import auto_scales
from pde_nondim.pinn import PINNFormulation

---
## Example 1 — Heat equation

$$\frac{\partial u}{\partial t} = \alpha \frac{\partial^2 u}{\partial x^2}$$

The natural time scale is diffusive: $t_c = L^2/\alpha$, which collapses the equation to a parameter-free form (Fourier number = 1).

In [2]:
x, t = sp.symbols('x t', positive=True)
alpha, L, T_c, dT = sp.symbols('alpha L T_c DeltaT', positive=True)
u = sp.Function('u')(x, t)

pde = sp.Eq(sp.diff(u, t), alpha * sp.diff(u, x, 2))
print('Dimensional PDE:')
sp.pprint(pde)

Dimensional PDE:
                 2          
∂               ∂           
──(u(x, t)) = α⋅───(u(x, t))
∂t                2         
                ∂x          


In [3]:
result = NonDimensionalizer(pde, scales={u: dT, x: L, t: T_c}).run()
print(result)

NON-DIMENSIONALISATION RESULT

Substitutions  (dimensional = ref + scale x dimensionless):
  x  =  L*xs
  t  =  T_c*ts
  u(x, t)  =  DeltaT*us(xs, ts)

Non-dimensional PDE (raw, before normalisation):
  Eq(DeltaT*Derivative(us(xs, ts), ts)/T_c, DeltaT*alpha*Derivative(us(xs, ts), (xs, 2))/L**2)

Normalised by coefficient:  DeltaT/T_c

Non-dimensional PDE  [O(1) normalised]:
  Eq(Derivative(us(xs, ts), ts) - T_c*alpha*Derivative(us(xs, ts), (xs, 2))/L**2, 0)

Dimensionless groups:
  Fo  =  T_c*alpha/L**2  -- Fourier — diffusion time / imposed time

Dominant-balance analysis:
  * Coefficient Fo (Fourier — diffusion time / imposed time):  -> large when Fo >> 1 (term dominates);  -> 0 when Fo << 1 (term negligible)


In [4]:
# With diffusive time scale T_c = L^2/alpha, Fo = 1 and no groups remain
result_canonical = NonDimensionalizer(pde, scales={u: dT, x: L, t: L**2/alpha}).run()
print('Canonical form (T_c = L²/α):')
sp.pprint(result_canonical.nd_pde_simplified)
print('Groups:', result_canonical.dimensionless_groups)

Canonical form (T_c = L²/α):
                    2                 
 ∂                 ∂                  
───(us(xs, ts)) - ────(us(xs, ts)) = 0
∂ts                  2                
                  ∂xs                 
Groups: {}


---
## Example 2 — Burgers equation

$$\frac{\partial u}{\partial t} + u\frac{\partial u}{\partial x} = \nu \frac{\partial^2 u}{\partial x^2}$$

The convective time scale $t_c = L/U$ reveals the Reynolds number $Re = UL/\nu$.

In [5]:
x, t = sp.symbols('x t', positive=True)
nu, L, U = sp.symbols('nu L U', positive=True)
u = sp.Function('u')(x, t)

pde = sp.Eq(sp.diff(u, t) + u*sp.diff(u, x), nu*sp.diff(u, x, 2))
print('Dimensional PDE:')
sp.pprint(pde)

Dimensional PDE:
                                       2          
        ∂             ∂               ∂           
u(x, t)⋅──(u(x, t)) + ──(u(x, t)) = ν⋅───(u(x, t))
        ∂x            ∂t                2         
                                      ∂x          


In [6]:
result = NonDimensionalizer(pde, scales={u: U, x: L, t: L/U}).run()
print(result)

NON-DIMENSIONALISATION RESULT

Substitutions  (dimensional = ref + scale x dimensionless):
  x  =  L*xs
  t  =  L*ts/U
  u(x, t)  =  U*us(xs, ts)

Non-dimensional PDE (raw, before normalisation):
  Eq(U**2*us(xs, ts)*Derivative(us(xs, ts), xs)/L + U**2*Derivative(us(xs, ts), ts)/L, U*nu*Derivative(us(xs, ts), (xs, 2))/L**2)

Normalised by coefficient:  U**2/L

Non-dimensional PDE  [O(1) normalised]:
  Eq(us(xs, ts)*Derivative(us(xs, ts), xs) + Derivative(us(xs, ts), ts) - nu*Derivative(us(xs, ts), (xs, 2))/(L*U), 0)

Dimensionless groups:
  1/Re  =  nu/(L*U)  -- Reynolds — inertia / viscous

Dominant-balance analysis:
  * Coefficient 1/Re (Reynolds — inertia / viscous):  -> 0 when Re >> 1 (term negligible);  -> large when Re << 1 (term dominates)


In [7]:
# Check magnitudes for water at 1 m/s over 0.1 m
result.check_magnitudes({nu: 1e-6, L: 0.1, U: 1.0})

'-----------------------------------------------------------------\nMAGNITUDE CHECK  (threshold = 10.0)\n  A group is O(1) if  1/10 <= value <= 10\n-----------------------------------------------------------------\n  1/Re          SMALL  (1e-05)  << 1  — term negligible, consider dropping\n\n  Tip: choose scales so that the dominant group equals 1.\n  Use suggest_scales() to find the balancing scale automatically.\n-----------------------------------------------------------------'

---
## Example 3 — Advection-diffusion

$$\frac{\partial u}{\partial t} + V\frac{\partial u}{\partial x} = D\frac{\partial^2 u}{\partial x^2}$$

Two natural time scales exist: convective $L/V$ and diffusive $L^2/D$.
The package handles both via `MultipleScalings`.

In [8]:
x, t = sp.symbols('x t', positive=True)
D, V, L, C0 = sp.symbols('D V L C_0', positive=True)
u = sp.Function('u')(x, t)

pde = sp.Eq(sp.diff(u, t) + V*sp.diff(u, x), D*sp.diff(u, x, 2))

ms = MultipleScalings(
    pde=pde,
    scale_options=[
        {u: C0, x: L, t: L/V},      # convective time scale -> Peclet number
        {u: C0, x: L, t: L**2/D},   # diffusive time scale  -> no groups
    ],
    labels=['convective', 'diffusive'],
)
results = ms.run_all()
for r in results:
    print(r)
    print()

('convective', NondimResult(nd_pde=Eq(C_0*V*Derivative(us(xs, ts), ts)/L + C_0*V*Derivative(us(xs, ts), xs)/L, C_0*D*Derivative(us(xs, ts), (xs, 2))/L**2), nd_pde_simplified=Eq(Derivative(us(xs, ts), (xs, 2)) - L*V*Derivative(us(xs, ts), ts)/D - L*V*Derivative(us(xs, ts), xs)/D, 0), dimensionless_groups={'Pe': L*V/D}, substitution_map={x: L*xs, t: L*ts/V, u(x, t): C_0*us(xs, ts)}, nonlinear_substitutions={}, reference_coefficient=-C_0*D/L**2, dominant_balance=['Coefficient Pe (Péclet (mass) — convection / diffusion):  -> large when Pe >> 1 (term dominates);  -> 0 when Pe << 1 (term negligible)']))

('diffusive', NondimResult(nd_pde=Eq(C_0*D*Derivative(us(xs, ts), ts)/L**2 + C_0*V*Derivative(us(xs, ts), xs)/L, C_0*D*Derivative(us(xs, ts), (xs, 2))/L**2), nd_pde_simplified=Eq(Derivative(us(xs, ts), ts) - Derivative(us(xs, ts), (xs, 2)) + L*V*Derivative(us(xs, ts), xs)/D, 0), dimensionless_groups={'Pe': L*V/D}, substitution_map={x: L*xs, t: L**2*ts/D, u(x, t): C_0*us(xs, ts)}, nonlinear_s

---
## Example 4 — Nonlinear diffusion with `k(u)`

$$\frac{\partial u}{\partial t} = \frac{\partial}{\partial x}\left(k(u)\frac{\partial u}{\partial x}\right)$$

Expanding the divergence gives two terms:
$$= k(u)\frac{\partial^2 u}{\partial x^2} + k'(u)\left(\frac{\partial u}{\partial x}\right)^2$$

Both are handled correctly with `nonlinear_scales`.

In [9]:
x, t = sp.symbols('x t', positive=True)
U, L, T_c, k_c = sp.symbols('U L T_c k_c', positive=True)
u = sp.Function('u')(x, t)
k = sp.Function('k')

pde = sp.Eq(sp.diff(u, t), sp.diff(k(u)*sp.diff(u, x), x))
print('Dimensional PDE:')
sp.pprint(pde)
print()
print('Expands to: k(u)u_xx + k\u2019(u)u_x²')

Dimensional PDE:
                          2                                               2
∂                        ∂                 ∂                 ⎛∂          ⎞ 
──(u(x, t)) = k(u(x, t))⋅───(u(x, t)) + ────────(k(u(x, t)))⋅⎜──(u(x, t))⎟ 
∂t                         2            ∂u(x, t)             ⎝∂x         ⎠ 
                         ∂x                                                

Expands to: k(u)u_xx + k’(u)u_x²


In [10]:
result = NonDimensionalizer(
    pde,
    scales={u: U, x: L, t: T_c},
    nonlinear_scales={k(u): k_c},
).run()
print(result)

NON-DIMENSIONALISATION RESULT

Substitutions  (dimensional = ref + scale x dimensionless):
  x  =  L*xs
  t  =  T_c*ts
  u(x, t)  =  U*us(xs, ts)

Nonlinear function scalings:
  k(u)  ->  k_c*ks(us(xs, ts))

Non-dimensional PDE (raw, before normalisation):
  Eq(U*Derivative(us(xs, ts), ts)/T_c, U*k_c*ks(us(xs, ts))*Derivative(us(xs, ts), (xs, 2))/L**2 + U*k_c*Derivative(ks(us(xs, ts)), us(xs, ts))*Derivative(us(xs, ts), xs)**2/L**2)

Normalised by coefficient:  U/T_c

Non-dimensional PDE  [O(1) normalised]:
  Eq(Derivative(us(xs, ts), ts) - T_c*k_c*ks(us(xs, ts))*Derivative(us(xs, ts), (xs, 2))/L**2 - T_c*k_c*Derivative(ks(us(xs, ts)), us(xs, ts))*Derivative(us(xs, ts), xs)**2/L**2, 0)

Dimensionless groups:
  1/Pe_T  =  T_c*k_c/L**2  -- Péclet (mass) — convection / diffusion

Dominant-balance analysis:
  * Coefficient 1/Pe (Péclet (mass) — convection / diffusion):  -> 0 when Pe >> 1 (term negligible);  -> large when Pe << 1 (term dominates)


In [11]:
# Confirm both k(u)*u_xx and k'(u)*u_x^2 terms are present
nd = result.nd_pde_simplified
terms = sp.Add.make_args(sp.expand(nd.lhs - nd.rhs))
print(f'Number of terms: {len(terms)}  (expect 3: time, k*u_xx, k\'*u_x²)')
for i, term in enumerate(terms, 1):
    print(f'  Term {i}:', term)

Number of terms: 3  (expect 3: time, k*u_xx, k'*u_x²)
  Term 1: -T_c*k_c*Derivative(ks(us(xs, ts)), us(xs, ts))*Derivative(us(xs, ts), xs)**2/L**2
  Term 2: -T_c*k_c*ks(us(xs, ts))*Derivative(us(xs, ts), (xs, 2))/L**2
  Term 3: Derivative(us(xs, ts), ts)


---
## Example 5 — LPBF 3-D Goldak heat source with hatch spacing

### The physical problem

Laser Powder Bed Fusion (LPBF) — a metal 3-D printing process. A laser scans across a powder bed, melting and fusing metal powder.

**Governing PDE** (3-D transient heat conduction):
$$\rho C_p \frac{\partial T}{\partial t} = k\left(\frac{\partial^2 T}{\partial x^2} + \frac{\partial^2 T}{\partial y^2} + \frac{\partial^2 T}{\partial z^2}\right) + Q(x,y,z,t)$$

**Goldak double-ellipsoid heat source**:
$$Q = \frac{6\sqrt{3}\,\eta P}{\pi^{3/2}\,a\,b\,c}\exp\left(-\frac{3x^2}{a^2} - \frac{3y^2}{b^2} - \frac{3(z-vt)^2}{c^2}\right)$$

where $a$, $b$, $c$ are the Goldak semi-axes, $v$ is scan speed, $\eta$ is absorptivity, $P$ is laser power.

**Additional parameter**: $h$ = hatch spacing (distance between adjacent scan tracks).

In [12]:
x, y, z, t = sp.symbols('x y z t')
rho, Cp, k  = sp.symbols('rho C_p k',  positive=True)
P, eta, v   = sp.symbols('P eta v',    positive=True)
a, b, c_g   = sp.symbols('a b c',      positive=True)   # Goldak semi-axes
h           = sp.Symbol('h',           positive=True)   # hatch spacing
T0          = sp.Symbol('T_0',         positive=True)
T = sp.Function('T')(x, y, z, t)

# Natural temperature scale: absorbs the source amplitude entirely
DeltaT = 6*sp.sqrt(3)*eta*P*a / (sp.pi**sp.Rational(3,2)*b*c_g*k)

# Goldak source
Q_peak = 6*sp.sqrt(3)*eta*P / (sp.pi**sp.Rational(3,2)*a*b*c_g)
q_src  = Q_peak * sp.exp(-3*(x**2/a**2 + y**2/b**2 + (z - v*t)**2/c_g**2))

pde = sp.Eq(rho*Cp*sp.diff(T, t),
            k*(sp.diff(T,x,2) + sp.diff(T,y,2) + sp.diff(T,z,2)) + q_src)

print('Dimensional PDE:')
sp.pprint(pde)

Dimensional PDE:
                                                 2      2      2               ↪
                                     3⋅(-t⋅v + z)    3⋅y    3⋅x                ↪
                                   - ───────────── - ──── - ────               ↪
                                           2           2      2                ↪
                                          c           b      a       ⎛ 2       ↪
     ∂                   6⋅√3⋅P⋅η⋅ℯ                                  ⎜∂        ↪
Cₚ⋅ρ⋅──(T(x, y, z, t)) = ─────────────────────────────────────── + k⋅⎜───(T(x, ↪
     ∂t                                 3/2                          ⎜  2      ↪
                                       π   ⋅a⋅b⋅c                    ⎝∂x       ↪

↪                                                      
↪                                                      
↪                                                      
↪                                                      
↪               2            

### Scaling choices

| Variable | Scale | Rationale |
|----------|-------|-----------|
| $x$ | $a$ | Goldak cross-track semi-axis → unit Gaussian in $\hat{x}$ |
| $y$ | $h$ | Hatch spacing → domain $\hat{y} \in [-\tfrac{1}{2}, \tfrac{1}{2}]$, one period |
| $z$ | $c$ | Goldak scan semi-axis → unit Gaussian in $\hat{z}$ |
| $t$ | $c/v$ | Convective (laser transit) time |
| $T$ | $T_0 + \Delta T$ | $\Delta T = 6\sqrt{3}\eta P a/(\pi^{3/2}bck)$ absorbs source amplitude |

In [13]:
scales = {T: (DeltaT, T0), x: a, y: h, z: c_g, t: c_g/v}
result = NonDimensionalizer(pde, scales=scales).run()
print(result)

NON-DIMENSIONALISATION RESULT

Substitutions  (dimensional = ref + scale x dimensionless):
  x  =  a*xs
  y  =  h*ys
  z  =  c*zs
  t  =  c*ts/v
  T(x, y, z, t)  =  6*sqrt(3)*P*a*eta*Ts(xs, ys, zs, ts)/(pi**(3/2)*b*c*k) + T_0

Non-dimensional PDE (raw, before normalisation):
  Eq(6*sqrt(3)*C_p*P*a*eta*rho*v*Derivative(Ts(xs, ys, zs, ts), ts)/(pi**(3/2)*b*c**2*k), 6*sqrt(3)*P*a*eta*Derivative(Ts(xs, ys, zs, ts), (ys, 2))/(pi**(3/2)*b*c*h**2) + 6*sqrt(3)*P*a*eta*Derivative(Ts(xs, ys, zs, ts), (zs, 2))/(pi**(3/2)*b*c**3) + 6*sqrt(3)*P*eta*Derivative(Ts(xs, ys, zs, ts), (xs, 2))/(pi**(3/2)*a*b*c) + 6*sqrt(3)*P*eta*exp(-3*ts**2)*exp(-3*xs**2)*exp(-3*zs**2)*exp(6*ts*zs)*exp(-3*h**2*ys**2/b**2)/(pi**(3/2)*a*b*c))

Normalised by coefficient:  6*sqrt(3)*C_p*P*a*eta*rho*v/(pi**(3/2)*b*c**2*k)

Non-dimensional PDE  [O(1) normalised]:
  Eq(Derivative(Ts(xs, ys, zs, ts), ts) - c*k*Derivative(Ts(xs, ys, zs, ts), (ys, 2))/(C_p*h**2*rho*v) - k*Derivative(Ts(xs, ys, zs, ts), (zs, 2))/(C_p*c*rho*v) - c*

### Moving frame + steady state

Transform to the co-moving frame $\xi = \hat{z} - \hat{t}$ (laser fixed at $\xi = 0$), then drop $\partial\hat{T}/\partial\hat{t}$ (quasi-steady melt pool).

In [14]:
zs = sp.Symbol('zs', positive=True)
ts = sp.Symbol('ts', positive=True)

# Named dimensionless groups
Pe_c    = sp.Symbol('Pe_c',    positive=True)   # rho*Cp*v*c/k
delta_x = sp.Symbol('delta_x', positive=True)   # a/c
delta_h = sp.Symbol('delta_h', positive=True)   # h/c
gamma   = sp.Symbol('gamma',   positive=True)   # h/b

pf = (
    PINNFormulation.from_result(result)
    .moving_frame(zs, ts, new_name='xi')
    .steady_state()
    .express_as_groups({
        '1/Pe_T':   sp.Integer(1) / (Pe_c * delta_h**2),
        '1/Pe_T_2': sp.Integer(1) / Pe_c,
        '1/Pe_T_3': sp.Integer(1) / (Pe_c * delta_x**2),
    })
)

# Substitute h/b -> gamma in the source term
residual = pf._residual.subs(h**2/b**2, gamma**2)
pf = pf._copy(_residual=residual)

print('Final non-dimensional PDE (moving frame, steady state):')
sp.pprint(sp.Eq(pf._residual, 0))

Final non-dimensional PDE (moving frame, steady state):
                           2                         2                         ↪
                          ∂                         ∂                          ↪
                          ───(Ts(xs, ys, ξ, ts))   ────(Ts(xs, ys, ξ, ts))     ↪
                            2                         2                        ↪
  ∂                       ∂ξ                       ∂xs                       ℯ ↪
- ──(Ts(xs, ys, ξ, ts)) - ────────────────────── - ─────────────────────── - ─ ↪
  ∂ξ                               Pe_c                          2             ↪
                                                          Pe_c⋅δₓ              ↪

↪                              2                        
↪                             ∂                         
↪     2       2      2   2   ────(Ts(xs, ys, ξ, ts))    
↪ -3⋅ξ   -3⋅xs   -3⋅γ ⋅ys       2                       
↪      ⋅ℯ      ⋅ℯ            ∂ys                        
↪ ────────

### Physical interpretation of the four groups

| Group | Expression | Meaning |
|-------|-----------|--------|
| $Pe_c$ | $\rho C_p v c / k$ | Scan Péclet — advection vs diffusion along laser path |
| $\delta_x = a/c$ | cross-track / scan semi-axis | Beam aspect ratio in $x$ |
| $\delta_h = h/c$ | hatch / scan semi-axis | Track pitch relative to beam length |
| $\gamma = h/b$ | hatch / depth semi-axis | Thermal overlap between adjacent tracks |

**Term-by-term:**
- $-\partial\hat{T}/\partial\xi$ — advection (O(1) by construction)
- $-(1/Pe_c)\,\partial^2\hat{T}/\partial\xi^2$ — scan-direction diffusion; small when $Pe_c \gg 1$
- $-(\delta_x^2/Pe_c)\,\partial^2\hat{T}/\partial\hat{x}^2$ — cross-track diffusion
- $-(1/Pe_c\delta_h^2)\,\partial^2\hat{T}/\partial\hat{y}^2$ — hatch-direction diffusion
- $-(\delta_x^2/Pe_c)\,e^{-3\xi^2-3\hat{x}^2-3\gamma^2\hat{y}^2}$ — Goldak source (unit Gaussian at $\xi=0$)

In [15]:
# Numerical values for 316L stainless steel
# rho=7990 kg/m3, Cp=500 J/kgK, k=15 W/mK
# P=200W, eta=0.35, v=0.8 m/s
# a=b=50um, c=150um, h=100um

import math
rho_v, Cp_v, k_v = 7990, 500, 15.0
P_v, eta_v, v_v  = 200, 0.35, 0.8
a_v, b_v, c_v, h_v = 50e-6, 50e-6, 150e-6, 100e-6

Pe_c_v    = rho_v * Cp_v * v_v * c_v / k_v
delta_x_v = a_v / c_v
delta_h_v = h_v / c_v
gamma_v   = h_v / b_v
DeltaT_v  = 6*math.sqrt(3)*eta_v*P_v*a_v / (math.pi**1.5 * b_v * c_v * k_v)

print('316L stainless steel — typical LPBF conditions')
print(f'  Pe_c   = {Pe_c_v:.1f}   (high Péclet → advection dominates)')
print(f'  delta_x = {delta_x_v:.2f}  (a/c, beam is wider than it is long)')
print(f'  delta_h = {delta_h_v:.2f}  (h/c, hatch is tighter than beam length)')
print(f'  gamma   = {gamma_v:.1f}    (h/b, tracks overlap significantly)')
print(f'  DeltaT  = {DeltaT_v:.0f} K  (characteristic temperature rise)')

316L stainless steel — typical LPBF conditions
  Pe_c   = 32.0   (high Péclet → advection dominates)
  delta_x = 0.33  (a/c, beam is wider than it is long)
  delta_h = 0.67  (h/c, hatch is tighter than beam length)
  gamma   = 2.0    (h/b, tracks overlap significantly)
  DeltaT  = 58063 K  (characteristic temperature rise)


### Generate PyTorch PINN residual code

In [16]:
code = (
    pf
    .set_domain(xs=(-3.0, 3.0), ys=(-0.5, 0.5), xi=(-3.0, 3.0))
    .set_parameters(Pe_c=(1.0, 50.0), delta_x=(0.3, 2.0),
                    delta_h=(0.5, 5.0), gamma=(0.5, 5.0))
    .pytorch_code()
)
print(code)

import math
import torch


def pde_residual(xs, ys, xi, Pe_c, delta_x, delta_h, gamma, model):
    """
    PDE residual — returns zero when the PDE is exactly satisfied.

    Inputs
    ------
    xs          : torch.Tensor  — dimensionless coord  (-3.0, 3.0)
    ys          : torch.Tensor  — dimensionless coord  (-0.5, 0.5)
    xi          : torch.Tensor  — dimensionless coord  (-3.0, 3.0)
    Pe_c        : float or Tensor  — (1.0, 50.0)
    delta_x     : float or Tensor  — (0.3, 2.0)
    delta_h     : float or Tensor  — (0.5, 5.0)
    gamma       : float or Tensor  — (0.5, 5.0)
    model       : nn.Module  — maps inputs to Ts
    """
    # Enable gradient computation for coordinates
    xs = xs.requires_grad_(True)
    ys = ys.requires_grad_(True)
    xi = xi.requires_grad_(True)

    # --- Map coordinates to [-1, 1] ---
    xs_n = 2.0 * (xs - (-3.0)) / (3.0 - (-3.0)) - 1.0
    ys_n = 2.0 * (ys - (-0.5)) / (0.5 - (-0.5)) - 1.0
    xi_n = 2.0 * (xi - (-3.0)) / (3.0 - (-3.0)) - 1.0

  

The generated `pde_residual` function:
- Takes **7 inputs**: 3 spatial coordinates + 4 process groups
- Normalises coordinates to `[-1, 1]` and parameters to `[0, 1]` before the network
- Uses `torch.autograd.grad` for all spatial derivatives
- Returns the residual `R` — use `loss = (R**2).mean()` in training

A single trained network covers the **entire process parameter space** — no re-training per material or machine condition.

---
## Bonus — Automatic scale discovery with `auto_scales`

Don't know which scales to use? Supply the **SI units** of each parameter — exactly what you already know — and let Buckingham Pi find the best scales automatically.

```
auto_scales(pde, dims, numerical_values={...})
```

- `dims` maps each symbol to its SI unit string: `'kg/m^3'`, `'W/m/K'`, `'J/kg/K'`, `'m/s'`, etc.
- `numerical_values` (optional) lets the package rank candidates by how balanced the resulting PDE is — score 0.0 means all terms are O(1).

No need to know M/L³ notation.

In [17]:
import sympy as sp
from pde_nondim.autoscale import auto_scales

x, t = sp.symbols('x t', positive=True)
rho, Cp, k, L, dT = sp.symbols('rho C_p k L DeltaT', positive=True)
T = sp.Function('T')(x, t)

pde_heat = sp.Eq(rho * Cp * sp.diff(T, t), k * sp.diff(T, x, 2))

# ── Supply SI units — exactly what you already know ────────────────────────
dims = {
    T:   'K',         # temperature
    x:   'm',         # length
    t:   's',         # time
    rho: 'kg/m^3',    # density
    Cp:  'J/kg/K',    # specific heat
    k:   'W/m/K',     # thermal conductivity
    L:   'm',         # domain length
    dT:  'K',         # temperature scale
}

# Numerical values for 316L stainless steel — used to rank candidates
numerical_values = {rho: 7990, Cp: 500, k: 15, L: 0.01, dT: 1000}

candidates = auto_scales(pde_heat, dims, numerical_values=numerical_values)

print(f'Found {len(candidates)} candidate scale sets (ranked by balance score)\n')
print(f'  {"Rank":>4}  {"Score":>6}  {"Time scale":>30}  Groups')
print('  ' + '-'*70)
for c in candidates[:5]:
    t_scale = [s for v, s in c['scales'].items() if str(v) == 't']
    t_str   = str(sp.simplify(t_scale[0])) if t_scale else '—'
    g_str   = str(c['groups']) if c['groups'] else 'none (perfectly balanced)'
    print(f"  {c['rank']:>4}  {c['score']:>6.3f}  {t_str:>30}  {g_str}")

Found 8 candidate scale sets (ranked by balance score)

  Rank   Score                      Time scale  Groups
  ----------------------------------------------------------------------
     1   0.000                  C_p*L**2*rho/k  none (perfectly balanced)
     2   0.000           k/(C_p**2*DeltaT*rho)  none (perfectly balanced)
     3   0.000                  C_p*L**2*rho/k  none (perfectly balanced)
     4   0.000           k/(C_p**2*DeltaT*rho)  none (perfectly balanced)
     5   2.121      L/(sqrt(C_p)*sqrt(DeltaT))  {'1/Pe_T': k/(C_p**(3/2)*sqrt(DeltaT)*L*rho)}


The rank-1 candidate has score 0.0 — the diffusive time scale $t_c = L^2 \rho C_p / k$ makes every coefficient exactly O(1). Feed it straight into `NonDimensionalizer`:

In [18]:
from pde_nondim import NonDimensionalizer

# Take the best candidate and run it directly
best = candidates[0]
print('Best scales:')
for var, scale in best['scales'].items():
    print(f'  {var}  ->  {scale}')
print()

result = NonDimensionalizer(pde_heat, scales=best['scales']).run()
print(result)

Best scales:
  T(x, t)  ->  (DeltaT, 0)
  x  ->  L
  t  ->  C_p*L**2*rho/k

NON-DIMENSIONALISATION RESULT

Substitutions  (dimensional = ref + scale x dimensionless):
  x  =  L*xs
  t  =  C_p*L**2*rho*ts/k
  T(x, t)  =  DeltaT*Ts(xs, ts)

Non-dimensional PDE (raw, before normalisation):
  Eq(DeltaT*k*Derivative(Ts(xs, ts), ts)/L**2, DeltaT*k*Derivative(Ts(xs, ts), (xs, 2))/L**2)

Normalised by coefficient:  DeltaT*k/L**2

Non-dimensional PDE  [O(1) normalised]:
  Eq(Derivative(Ts(xs, ts), ts) - Derivative(Ts(xs, ts), (xs, 2)), 0)


### Burgers equation — auto scale discovery with SI units

In [19]:
x, t = sp.symbols('x t', positive=True)
nu, L, U = sp.symbols('nu L U', positive=True)
u = sp.Function('u')(x, t)

pde_burgers = sp.Eq(sp.diff(u, t) + u*sp.diff(u, x), nu*sp.diff(u, x, 2))

dims_burgers = {
    u:  'm/s',      # velocity
    x:  'm',        # length
    t:  's',        # time
    nu: 'm^2/s',    # kinematic viscosity
    L:  'm',        # length scale
    U:  'm/s',      # velocity scale
}

candidates_b = auto_scales(
    pde_burgers,
    dims_burgers,
    numerical_values={nu: 1e-6, L: 0.1, U: 1.0},
)

print('Burgers equation — top candidates:\n')
for c in candidates_b[:4]:
    t_scale = [s for v, s in c['scales'].items() if str(v) == 't']
    t_str   = str(sp.simplify(t_scale[0])) if t_scale else '—'
    g_str   = str(c['groups']) if c['groups'] else 'none'
    print(f"  Rank {c['rank']}  score={c['score']:.3f}  t={t_str}  groups={g_str}")

Burgers equation — top candidates:

  Rank 1  score=1.732  t=nu/U**2  groups=none
  Rank 2  score=1.732  t=L**2/nu  groups=none
  Rank 3  score=1.732  t=nu/U**2  groups={'1/Re': nu/(L*U)}
  Rank 4  score=2.449  t=nu/U**2  groups={'1/Re': nu/(L*U), '1/Re_2': nu**2/(L**2*U**2)}
